# Entrega 01: Relatório de Análise de Dados e Plano do Problema
## Projeto: Inteligência Artificial e Visão Computacional para Diagnóstico Precoce da Doença de Lyme
**Instituição:** FESA - Faculdade Engenheiro Salvador Arena  
**Curso:** Engenharia de Computação  
**Disciplina:** Análise de Dados (N1 - 2º Bimestre)  
**Dataset:** Lyme Disease Rashes (Full Stratified - Kaggle)  
**Autor:** Murilo Trevejo  
**Data de Entrega:** 19/10/2026

---
### Sumário do Estudo
Este notebook executa a pipeline completa da **Análise Exploratória de Dados Preliminar (EDA)**, extração de metadados espectrais e geométricos, e diagnóstico de integridade da base fotográfica de lesões de **Doença de Lyme (Eritema Migratório)** e 12 diagnósticos diferenciais dermatológicos.

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Configurações estéticas
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 10

print('Bibliotecas importadas com sucesso!')

## 1. Carregamento dos Metadados Estruturados
Carregamos a base de dados tabulares estruturada gerada a partir das 5.059 imagens dermatológicas.

In [ ]:
csv_path = '../data/metadata_lyme_rashes.csv'
df = pd.read_csv(csv_path)
print(f'Total de instâncias fotográficas analisadas: {len(df):,}')
df.head()

## 2. Diagnóstico de Dados Faltantes e Integridade
Verificação quantitativa de valores nulos e validação de cabeçalhos de arquivos.

In [ ]:
missing = df.isnull().sum()
print('Contagem de valores ausentes por variável:')
print(missing)
print(f'\nImagens corrompidas no disco: {df["corrupted"].sum()} (0.0%)')

## 3. Análise de Desbalanceamento de Classes e Partições
Investigação da distribuição entre casos Positivos (Eritema de Lyme) e Negativos (Outras lesões).

In [ ]:
class_counts = df['binary_class'].value_counts()
print('Distribuição Absoluta e Percentual:')
for cls, count in class_counts.items():
    print(f' - {cls}: {count} imagens ({count/len(df)*100:.2f}%)')

ratio = class_counts['Negative'] / class_counts['Positive']
print(f'\nRazão de Desbalanceamento (Negativo : Positivo): {ratio:.2f} : 1')

print('\nDistribuição por Split Oficial (Treino / Validação / Teste):')
display(pd.crosstab(df['split'], df['binary_class'], margins=True))

## 4. Detalhamento das 15 Condições Clínicas
Mapeamento taxonômico das variantes de Lyme e dos 12 diagnósticos diferenciais.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
cond_counts = df['condition'].value_counts()
colors = ['#DC2626' if c in ['Erythema Migrans', 'bullseye rash', 'EM rash'] else '#3B82F6' for c in cond_counts.index]
sns.barplot(x=cond_counts.values, y=cond_counts.index, palette=colors, ax=ax)
ax.set_title('Distribuição das 15 Condições Dermatológicas no Dataset', fontsize=13, weight='bold')
ax.set_xlabel('Quantidade de Imagens')
ax.set_ylabel('Patologia / Condição Clínica')
for i, v in enumerate(cond_counts.values):
    ax.text(v + 5, i, str(v), va='center', fontweight='bold', fontsize=9)
plt.tight_layout()
plt.show()

## 5. Dimensões Espaciais e Resolução (Outliers Geométricos)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=df, x='width', y='height', hue='binary_class', alpha=0.5,
                palette={'Negative': '#6B7280', 'Positive': '#DC2626'}, ax=axes[0])
axes[0].set_title('Largura vs Altura (pixels)', fontsize=12, weight='bold')
axes[0].set_xlabel('Largura (px)')
axes[0].set_ylabel('Altura (px)')

sns.histplot(data=df, x='aspect_ratio', hue='binary_class', bins=35, kde=True,
             palette={'Negative': '#6B7280', 'Positive': '#DC2626'}, ax=axes[1])
axes[1].set_title('Distribuição do Aspect Ratio (Largura / Altura)', fontsize=12, weight='bold')
axes[1].axvline(1.0, color='red', linestyle='--', label='1:1 (Quadrada)')
axes[1].legend()
plt.tight_layout()
plt.show()

## 6. Análise Espectral de Cores e Índice de Eritema (R - G)
Avaliação do poder discriminatório da hiperemia cutânea isolada.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

rgb_means = pd.DataFrame({
    'Canal R': [df[df['binary_class']=='Positive']['mean_r'].mean(), df[df['binary_class']=='Negative']['mean_r'].mean()],
    'Canal G': [df[df['binary_class']=='Positive']['mean_g'].mean(), df[df['binary_class']=='Negative']['mean_g'].mean()],
    'Canal B': [df[df['binary_class']=='Positive']['mean_b'].mean(), df[df['binary_class']=='Negative']['mean_b'].mean()],
}, index=['Positivo (Lyme)', 'Negativo (Outros)'])

rgb_means.plot(kind='bar', ax=axes[0], color=['#EF4444', '#10B981', '#3B82F6'], width=0.6)
axes[0].set_title('Intensidade Média por Canal de Cor (RGB)', fontsize=12, weight='bold')
axes[0].set_ylabel('Intensidade (0-255)')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

sns.kdeplot(data=df, x='erythema_index', hue='binary_class', fill=True, common_norm=False,
            palette={'Negative': '#6B7280', 'Positive': '#DC2626'}, ax=axes[1], alpha=0.4)
axes[1].set_title('Densidade do Índice de Eritema Cutâneo (R - G)', fontsize=12, weight='bold')
axes[1].set_xlabel('Índice de Eritema (R - G)')
plt.tight_layout()
plt.show()

## 7. Detecção de Outliers e Ruído Óptico (Brilho e Contraste)
Identificação de amostras com subexposição, superexposição (flash especular) e desfoque.

In [ ]:
q_b_low, q_b_high = df['brightness'].quantile(0.01), df['brightness'].quantile(0.99)
q_c_low, q_c_high = df['contrast'].quantile(0.01), df['contrast'].quantile(0.99)

print(f'Limiares de Brilho (Percentil 1% e 99%): {q_b_low:.2f} e {q_b_high:.2f}')
print(f'Limiares de Contraste (Percentil 1% e 99%): {q_c_low:.2f} e {q_c_high:.2f}')

fig, ax = plt.subplots(figsize=(10, 5.5))
sns.scatterplot(data=df, x='brightness', y='contrast', hue='binary_class', alpha=0.5,
                palette={'Negative': '#6B7280', 'Positive': '#DC2626'}, ax=ax)
ax.axvline(q_b_low, color='red', linestyle='--', label=f'Outliers Brilho ({q_b_low:.1f}, {q_b_high:.1f})')
ax.axvline(q_b_high, color='red', linestyle='--')
ax.axhline(q_c_low, color='orange', linestyle='--', label=f'Outliers Contraste ({q_c_low:.1f}, {q_c_high:.1f})')
ax.axhline(q_c_high, color='orange', linestyle='--')
ax.set_title('Espaço de Brilho vs Contraste com Fronteiras de Outliers', fontsize=13, weight='bold')
ax.legend(loc='lower left')
plt.tight_layout()
plt.show()

## 8. As 4 Perguntas-Chave do Projeto (Problema de Análise de Dados e Aprendizado de Máquina)

### 1. Pergunta de Agrupamento (Clustering Não Supervisionado)
> *"Existem subgrupos, padrões morfológicos latentes ou fenótipos visuais não rotulados entre as lesões de eritema de pacientes com Lyme que permitam categorizar diferentes perfis de gravidade ou tempo de evolução clínica?"*

### 2. Pergunta de Classificação (Aprendizado Supervisionado)
> *"É possível construir um modelo preditivo baseado em redes neurais convolucionais capaz de diagnosticar precocemente o eritema da Doença de Lyme e diferenciá-lo de outras 12 patologias dermatológicas similares em meio a imagens ruidosas?"*

### 3. Pergunta Preditiva (Regressão / Modelagem Temporal)
> *"Como modelar e prever a dinâmica geométrica de expansão temporal da lesão (taxa de crescimento diametral em cm/dia) e estimar a probabilidade contínua de disseminação sistêmica ou transição para fases crônicas?"*

### 4. Pergunta de Suporte à Decisão (Aplicação Prescritiva / CDSS)
> *"Como as predições probabilísticas calibradas e mapas de explicabilidade visual (Grad-CAM) podem ser integrados em um Sistema de Suporte à Decisão Clínica (CDSS) para auxiliar na recomendação de condutas terapêuticas precoces e otimização de tratamento?"*

--- 
**Relatório Técnico Oficial Gerado:** `reports/Relatorio_Entrega_01_Doenca_de_Lyme.pdf`